# Colab Setup — TFM Pose 6-DoF

**Este notebook configura TODO el entorno en Google Colab.**

## Estrategia Local vs Colab

| Tarea | Local (M1 Pro) | Colab (T4/A100) |
|-------|---------------|------------------|
| Desarrollo de codigo | Editor + git | Lento para editar |
| Docker / ROS 2 / CoppeliaSim | Docker Desktop | No soportado |
| Notebooks matematicos | Jupyter local | Tambien funciona |
| **Datasets BOP (30+ GB)** | Disco limitado | ~80 GB libres |
| **FoundationPose inferencia** | Necesita CUDA | T4/A100 GPU |
| **GDR-Net inferencia** | Necesita CUDA | T4/A100 GPU |
| **Diffusion Policy training** | Muy lento en CPU | GPU acelerado |
| **BOP Toolkit evaluacion** | CPU suficiente | Tambien funciona |
| Visualizacion / figuras TFM | Mejor local | Inline |

### Flujo de trabajo recomendado:
1. **Desarrollar** en local (VSCode + git)
2. **Push** a GitHub
3. **Pull** en Colab y ejecutar evaluaciones GPU
4. **Descargar** resultados (.json, figuras) a local
5. **Integrar** en memoria TFM

## 0. Configuracion Global

Todas las constantes compartidas se definen aqui para evitar `NameError` si se ejecutan celdas individuales.

In [ ]:
# ============================================================
# CONFIGURACION GLOBAL — Ejecutar SIEMPRE primero
# ============================================================
import os

REPO_URL  = "https://github.com/Giocrisrai/pose6dof-transformers-diffusion.git"
REPO_DIR  = "/content/repo_tfm"
USE_GDRIVE = True   # Cambiar a False si no quieres persistencia en Drive

if USE_GDRIVE:
    DATA_ROOT   = "/content/drive/MyDrive/TFM/datasets"
    WEIGHTS_DIR = "/content/drive/MyDrive/TFM/weights"
else:
    DATA_ROOT   = f"{REPO_DIR}/data/datasets"
    WEIGHTS_DIR = f"{REPO_DIR}/weights"

print(f"REPO_DIR    = {REPO_DIR}")
print(f"DATA_ROOT   = {DATA_ROOT}")
print(f"WEIGHTS_DIR = {WEIGHTS_DIR}")
print(f"USE_GDRIVE  = {USE_GDRIVE}")

## 1. Verificar GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1e9:.1f} GB")
else:
    print("\nVe a Runtime > Change runtime type > T4 GPU")

# Espacio en disco
!df -h /content | tail -1
!free -h | head -2

## 2. Clonar repositorio TFM

In [ ]:
import os

if os.path.exists(REPO_DIR):
    print("Repo ya existe, haciendo pull...")
    !cd {REPO_DIR} && git pull
else:
    print("Clonando repositorio...")
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git log --oneline -5

## 3. Instalar dependencias

In [ ]:
# Colab ya tiene PyTorch + CUDA, solo instalamos las extras
!pip install -q open3d trimesh pyrender
!pip install -q git+https://github.com/thodan/bop_toolkit.git
!pip install -q diffusers accelerate transformers
!pip install -q scipy scikit-learn Pillow tqdm

# Verificar imports del repo
import sys
sys.path.insert(0, REPO_DIR)
from src.utils.lie_groups import so3_exp, se3_exp
from src.utils.rotations import matrix_to_6d, sixd_to_matrix
from src.utils.metrics import compute_add, compute_adds
from src.planning.diffusion_policy import DiffusionGraspPlanner
from src.pipeline import BinPickingPipeline, PipelineConfig
print("\nTodos los imports OK")

## 4. Descargar Datasets BOP

Colab tiene ~80 GB, suficiente para ambos datasets completos.

**Importante:** Los datasets se guardan en `/content/` (se borran al terminar la sesion).
Para persistencia, monta Google Drive.

In [ ]:
# ============================================================
# Montar Google Drive (si USE_GDRIVE=True)
# ============================================================
if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_ROOT, exist_ok=True)
    print(f"Datos se guardan en Google Drive: {DATA_ROOT}")
else:
    os.makedirs(DATA_ROOT, exist_ok=True)
    print(f"Datos temporales en: {DATA_ROOT} (se pierden al cerrar)")

# Symlink para que el codigo siempre encuentre los datos
REPO_DATA = f"{REPO_DIR}/data/datasets"
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA):
    !rm -rf {REPO_DATA}
os.makedirs(os.path.dirname(REPO_DATA), exist_ok=True)
os.symlink(DATA_ROOT, REPO_DATA)
print(f"Symlink: {REPO_DATA} -> {DATA_ROOT}")

In [ ]:
# ============================================================
# Descarga YCB-Video desde BOP (Hugging Face mirror)
# ============================================================
YCBV_DIR = f"{DATA_ROOT}/ycbv"
os.makedirs(YCBV_DIR, exist_ok=True)

HF_BASE = "https://huggingface.co/datasets/bop-benchmark"

ycbv_files = {
    "ycbv_base.zip": f"{HF_BASE}/ycbv/resolve/main/ycbv_base.zip",
    "ycbv_models.zip": f"{HF_BASE}/ycbv/resolve/main/ycbv_models.zip",
    "ycbv_test_all.zip": f"{HF_BASE}/ycbv/resolve/main/ycbv_test_all.zip",
}

for name, url in ycbv_files.items():
    dest = f"{YCBV_DIR}/{name}"
    # Comprobar si ya esta extraido
    extract_marker = name.replace(".zip", ".extracted")
    if os.path.exists(f"{YCBV_DIR}/{extract_marker}"):
        print(f"[OK] {name} ya procesado, saltando")
        continue
    print(f"\nDescargando {name}...")
    !wget -q --show-progress -c -O {dest} "{url}"
    print(f"Extrayendo {name}...")
    !cd {YCBV_DIR} && unzip -qo {dest} && rm {dest}
    # Marker para no re-descargar
    !touch {YCBV_DIR}/{extract_marker}
    print(f"[OK] {name} listo")

print(f"\n=== YCB-V contenido ===")
!ls {YCBV_DIR}/
!du -sh {YCBV_DIR}/

In [ ]:
# ============================================================
# Descarga T-LESS desde BOP
# ============================================================
TLESS_DIR = f"{DATA_ROOT}/tless"
os.makedirs(TLESS_DIR, exist_ok=True)

tless_files = {
    "tless_base.zip": f"{HF_BASE}/tless/resolve/main/tless_base.zip",
    "tless_models.zip": f"{HF_BASE}/tless/resolve/main/tless_models.zip",
    "tless_test_primesense_all.zip": f"{HF_BASE}/tless/resolve/main/tless_test_primesense_all.zip",
}

for name, url in tless_files.items():
    dest = f"{TLESS_DIR}/{name}"
    extract_marker = name.replace(".zip", ".extracted")
    if os.path.exists(f"{TLESS_DIR}/{extract_marker}"):
        print(f"[OK] {name} ya procesado, saltando")
        continue
    print(f"\nDescargando {name}...")
    !wget -q --show-progress -c -O {dest} "{url}"
    print(f"Extrayendo {name}...")
    !cd {TLESS_DIR} && unzip -qo {dest} && rm {dest}
    !touch {TLESS_DIR}/{extract_marker}
    print(f"[OK] {name} listo")

print(f"\n=== T-LESS contenido ===")
!ls {TLESS_DIR}/
!du -sh {TLESS_DIR}/

In [ ]:
# Resumen de espacio
print("=== Espacio en disco ===")
!df -h /content | tail -1
print("\n=== Datasets ===")
!du -sh {DATA_ROOT}/*/
print(f"\n=== Symlink verificado ===")
!ls -la {REPO_DIR}/data/datasets/

## 5. Verificar Dataset Loader

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
from src.utils.dataset_loader import BOPDataset
import matplotlib.pyplot as plt
import numpy as np

# Cargar YCB-V
try:
    ycbv = BOPDataset(f"{REPO_DIR}/data/datasets/ycbv", split="test")
    scenes = ycbv.get_scene_ids()
    print(f"YCB-V: {len(scenes)} escenas de test")
    
    # Cargar una muestra
    sample = ycbv.load_sample(scenes[0], 0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(sample['rgb']); ax1.set_title(f'YCB-V Scene {scenes[0]} - RGB')
    ax2.imshow(sample['depth'], cmap='viridis'); ax2.set_title('Depth')
    plt.tight_layout(); plt.show()
    print(f"  RGB shape: {sample['rgb'].shape}")
    print(f"  Depth shape: {sample['depth'].shape}")
    print(f"  Camera K:\n{sample['cam_K']}")
except Exception as e:
    print(f"YCB-V no cargado: {e}")

# Cargar T-LESS
try:
    tless = BOPDataset(f"{REPO_DIR}/data/datasets/tless", split="test_primesense")
    scenes = tless.get_scene_ids()
    print(f"\nT-LESS: {len(scenes)} escenas de test")
    
    sample = tless.load_sample(scenes[0], 0)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(sample['rgb']); ax1.set_title(f'T-LESS Scene {scenes[0]} - RGB')
    ax2.imshow(sample['depth'], cmap='viridis'); ax2.set_title('Depth')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f"T-LESS no cargado: {e}")

## 6. Descargar Pesos Pre-entrenados

In [ ]:
# ============================================================
# FoundationPose weights (NVIDIA)
# ============================================================
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if USE_GDRIVE:
    # Symlink para que el repo encuentre los pesos
    repo_weights = f"{REPO_DIR}/weights"
    if os.path.islink(repo_weights): os.unlink(repo_weights)
    elif os.path.isdir(repo_weights): 
        import shutil; shutil.rmtree(repo_weights)
    os.symlink(WEIGHTS_DIR, repo_weights)

FP_WEIGHTS = f"{WEIGHTS_DIR}/foundationpose"
os.makedirs(FP_WEIGHTS, exist_ok=True)

# FoundationPose usa modelos de NeRF + pose refiner
# Los pesos oficiales estan en el repo de NVIDIA
print("FoundationPose weights:")
print("Los pesos se descargan automaticamente al ejecutar el modelo.")
print(f"Directorio: {FP_WEIGHTS}")
print("\nPara descarga manual:")
print("  https://github.com/NVlabs/FoundationPose#pre-trained-weights")

In [ ]:
# ============================================================
# GDR-Net++ weights (BOP 2022 winner)
# ============================================================
GDRNET_WEIGHTS = f"{WEIGHTS_DIR}/gdrnet"
os.makedirs(GDRNET_WEIGHTS, exist_ok=True)

print("GDR-Net++ weights:")
print("  Pre-entrenados disponibles en:")
print("  https://github.com/shanice-l/gdrnpp_bop2022#model-zoo")
print(f"\nDirectorio: {GDRNET_WEIGHTS}")

## 7. Quick Sanity Check — GPU + Nuestro Codigo

In [ ]:
import torch
import numpy as np
import sys; sys.path.insert(0, REPO_DIR)
from src.utils.lie_groups import so3_exp, pose_from_Rt, geodesic_distance_SE3
from src.planning.diffusion_policy import DiffusionGraspPlanner, ConditionalUNet1D

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Test 1: UNet1D en GPU
model = ConditionalUNet1D(action_dim=7, horizon=16, cond_dim=64).to(device)
x = torch.randn(4, 16, 7, device=device)  # batch=4
t = torch.randint(0, 100, (4,), device=device)
c = torch.randn(4, 64, device=device)

with torch.no_grad():
    out = model(x, t, c)
print(f"UNet1D forward pass: {out.shape} on {device}")

# Test 2: Grasp planner en GPU
planner = DiffusionGraspPlanner(device=device)
R = so3_exp(np.array([0.1, -0.2, 0.3]))
T = pose_from_Rt(R, np.array([0.15, -0.08, 0.45]))
traj = planner.plan_grasp_heuristic(T)
print(f"Heuristic grasp: {traj.shape}")

# Test 3: Benchmark GPU throughput
if device == "cuda":
    import time
    batch = torch.randn(64, 16, 7, device=device)
    ts = torch.randint(0, 100, (64,), device=device)
    conds = torch.randn(64, 64, device=device)
    
    # Warmup
    for _ in range(10): model(batch, ts, conds)
    torch.cuda.synchronize()
    
    t0 = time.time()
    for _ in range(100): model(batch, ts, conds)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    print(f"GPU throughput: {100*64/elapsed:.0f} samples/s ({elapsed/100*1000:.1f} ms/batch)")

print(f"\nTodo funciona correctamente en Colab con {device.upper()}!")

## 8. Guardar resultados a Google Drive

Utility para guardar resultados de experimentos

In [ ]:
import json
from datetime import datetime

def save_experiment(name, results, figures=None):
    """Guardar resultados de experimento a Drive.
    
    Args:
        name: nombre del experimento (e.g. 'foundationpose_ycbv')
        results: dict con metricas
        figures: list of (fig, filename) tuples
    """
    exp_dir = f"{REPO_DIR}/experiments/{name}"
    if USE_GDRIVE:
        exp_dir = f"/content/drive/MyDrive/TFM/experiments/{name}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Guardar metricas
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    results['timestamp'] = timestamp
    results['gpu'] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    
    with open(f"{exp_dir}/results_{timestamp}.json", 'w') as f:
        json.dump(results, f, indent=2)
    print(f"Resultados guardados: {exp_dir}/results_{timestamp}.json")
    
    # Guardar figuras
    if figures:
        for fig, fname in figures:
            fig.savefig(f"{exp_dir}/{fname}", dpi=150, bbox_inches='tight')
            print(f"Figura: {exp_dir}/{fname}")
    
    return exp_dir

# Ejemplo
save_experiment('test_setup', {
    'status': 'Colab configurado correctamente',
    'pytorch': torch.__version__,
    'cuda': torch.cuda.is_available(),
})
print("\nSetup completo! Puedes continuar con los notebooks de evaluacion.")